# Lab 8.1 – Zahlen und Aussagen prüfen

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

Drei fertige Gegenproben zum Managementkommentar `m08-l01-kommentar_oktober.md`, danach eine Prüfzelle für den korrigierten Kommentar.

| Gegenprobe | prüft | Quelle |
| --- | --- | --- |
| 1 | Zahlen im Kommentar | Oktober-Bericht aus Lab 7.3 (`kpi_2026_10.xlsx`, Sollwert-Prüfung bestanden) |
| 2 | Aussage zum Relaunch des Onlineshops | Umsatz je Kanal April–Oktober 2026 gegen dieselben Monate 2025 |
| 3 | Aussage über die Regionen | Umsatz je Region Oktober 2026 gegen Oktober 2025 |

In [ ]:
import os
import re

import pandas as pd

ORDNER = "project/playground/"
KOMMENTAR = ORDNER + "m08-l01-kommentar_oktober.md"     # der Kommentar, der geprüft wird
BERICHT = "project/examples/m07-analysis-skill/l03-new-delivery/berichte/kpi_2026_10.xlsx"   # Oktober-Bericht (Lab 7.3)

OKTOBER = "data/nordwind_umsatz_2026_10.csv"                  # seit Lab 7.3 in data/
if not os.path.exists(OKTOBER):
    OKTOBER = "lieferungen/nordwind_umsatz_2026_10.csv"       # falls die Lieferung noch nicht verschoben ist

# Historie + Oktober-Lieferung, bereinigt wie in Lab 2.2 und im Bericht v2
roh = pd.concat([pd.read_csv("data/nordwind_umsatz.csv"), pd.read_csv(OKTOBER)], ignore_index=True)
df = roh.drop_duplicates().copy()
datum = pd.to_datetime(df["datum"], format="%d.%m.%Y", errors="coerce")
df["datum"] = datum.fillna(pd.to_datetime(df["datum"], format="%Y-%m-%d", errors="coerce"))
df["region"] = (df["region"].str.strip().str.replace("Region ", "", regex=False)
                .str.capitalize().replace({"Sued": "Süd"}))
df["monat"] = df["datum"].dt.strftime("%Y-%m")
print(len(df), "Zeilen nach Bereinigung, Monate", df["monat"].min(), "bis", df["monat"].max())

## Gegenprobe 1 (fertig): Zahlen aus dem Oktober-Bericht – Fehler (a)

Der Bericht aus Lab 7.3 ist die geprüfte Quelle. Jede Zahl im Kommentar muss sich hier wiederfinden.

In [ ]:
bericht = pd.read_excel(BERICHT, sheet_name="Kennzahlen", index_col=0)
bericht

## Gegenprobe 2 (fertig): Hat der Relaunch den Umsatz um 20 % gesteigert? – Fehler (b)

Relaunch des Onlineshops am 01.03.2026. Verglichen werden April–Oktober 2026 und **dieselben Monate 2025**: So zählt die Saison (Lab 6.1) nicht als Effekt. Getrennt nach Kanal zeigt sich, ob Umsatz neu dazukommt oder nur den Kanal wechselt.

In [ ]:
monate_2026 = [f"2026-{m:02d}" for m in range(4, 11)]          # April bis Oktober 2026
monate_2025 = [m.replace("2026", "2025") for m in monate_2026]  # dieselben Monate im Vorjahr

kanal = pd.DataFrame({
    "Apr–Okt 2025": df[df["monat"].isin(monate_2025)].groupby("kanal")["umsatz"].sum(),
    "Apr–Okt 2026": df[df["monat"].isin(monate_2026)].groupby("kanal")["umsatz"].sum(),
})
kanal.loc["Gesamt"] = kanal.sum()
kanal["Δ %"] = (kanal["Apr–Okt 2026"] / kanal["Apr–Okt 2025"] - 1) * 100
kanal["Anteil 2025 %"] = kanal["Apr–Okt 2025"] / kanal.loc["Gesamt", "Apr–Okt 2025"] * 100
kanal["Anteil 2026 %"] = kanal["Apr–Okt 2026"] / kanal.loc["Gesamt", "Apr–Okt 2026"] * 100
kanal.round(1)

## Gegenprobe 3 (fertig): Sind alle Regionen gewachsen? – Fehler (c)

Eine Gesamtzahl kann den Rückgang einer kleinen Region verdecken. Deshalb jede Region einzeln gegen den Vorjahresmonat.

In [ ]:
region = pd.DataFrame({
    "Okt 2025": df[df["monat"] == "2025-10"].groupby("region")["umsatz"].sum(),
    "Okt 2026": df[df["monat"] == "2026-10"].groupby("region")["umsatz"].sum(),
})
region["Δ Vorjahr %"] = (region["Okt 2026"] / region["Okt 2025"] - 1) * 100
region["Anteil Okt 2026 %"] = region["Okt 2026"] / region["Okt 2026"].sum() * 100
region.round(1)

## Prüfzelle: korrigierter Kommentar

Nach den Korrekturen den Kommentar speichern (**Strg+S**, macOS: Cmd+S) und diese Zelle erneut ausführen.

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_kommentar():
    if not os.path.exists(KOMMENTAR):
        return f"nicht OK – {KOMMENTAR} fehlt: Starter m08-l01-kommentar_oktober.md nach project/playground/ kopieren."
    text = open(KOMMENTAR, encoding="utf-8").read()
    zeilen = []
    if "373.139" in text or "337.139" not in text:
        zeilen.append("nicht OK – (a) Umsatz Oktober: Zahl im Text mit Gegenprobe 1 (Zeile Gesamt) vergleichen.")
    else:
        zeilen.append("OK – (a) Umsatz Oktober wie im Bericht.")
    if re.search(r"um\s*20\s*%", text) or "7,2" not in text:
        zeilen.append("nicht OK – (b) Relaunch: Aussage „um 20 %“ ersetzen und das Gesamtwachstum aus Gegenprobe 2 nennen.")
    else:
        zeilen.append("OK – (b) Relaunch-Aussage mit dem gemessenen Gesamtwachstum.")
    if "alle regionen" in text.lower() or "8,7" not in text:
        zeilen.append("nicht OK – (c) Regionen: „alle Regionen“ ersetzen und den Rückgang aus Gegenprobe 3 nennen.")
    else:
        zeilen.append("OK – (c) Regionen-Aussage nennt den Rückgang.")
    if all(z.startswith("OK") for z in zeilen):
        zeilen.append("OK – alle drei Korrekturen sind belegt.")
    return "\n".join(zeilen)


print(pruefe_kommentar())